# Data Warehouse — Sales Analytics
## Mini Project

| Aspek | Detail |
|---|---|
| Source | (1) CSV 2 tanggal (batch) · (2) MariaDB `stg_categories` (streaming) |
| Warehouse | Neon PostgreSQL — schema `stg` → `dwh` |
| Orkestrasi | Apache Airflow (DAG `dag_manuel`) |
| Streaming | Kafka + Debezium (CDC) |
| Dashboard | Metabase |
| **Total transaksi (`fact_sales`)** | **99.051** |

---
## 1. Scenario & Arsitektur

Sesuai skenario, ada **dua jalur data** yang bermuara ke Postgres (Neon) yang sama:

```
  (ATAS/BATCH)  CSV 2 tanggal --[Airflow]------+
                                               v
                                 Staging (stg) --> Fact & Dimensi (dwh) --> Metabase
                                               ^
  (BAWAH/STREAM) MariaDB stg_categories --[CDC: Debezium+Kafka]--+
```

- **Pipeline atas = Batch** (base pipeline): CSV → Airflow → `stg` → `dwh`.
- **Pipeline bawah = Streaming**: 1 tabel `stg_categories` dari MariaDB direplikasi real-time ke Postgres yang sama (seperti tugas streaming).
- Keduanya di **server yang sama**; hasil akhir divisualisasikan di Metabase.

---
## 2. Source Data — Dua Tanggal (CSV)

Sumber batch = file CSV pada **2 tanggal** (2018-05-08 & 2018-05-09). Isinya sama, beda tanggal:
7 tabel — `sales`, `customers`, `products`, `categories`, `cities`, `countries`, `employee`.

| Batch | Baris sales | Kumulatif `fact_sales` |
|---|---:|---:|
| 2018-05-08 (full) | 98.255 | 98.255 |
| 2018-05-09 (incremental) | +796 | **99.051** |

> Trigger ulang tanggal sama tidak menambah duplikat (idempotent).

---
## 3. Pipeline Batch — Airflow (stg → dwh)

CSV di-load Airflow ke schema `stg`, lalu dibentuk dimensi & fact di schema `dwh`.
Urutan task: `load_staging` → `dim_product` / `dim_customer` / `dim_employee` → `fact_sales`.

![Airflow DAG Sukses](docs/airflow.jpeg)

- **SCD**: `dim_product` Type 1, `dim_customer` & `dim_employee` Type 2, `dim_time` Type 0.
- **Idempotent**: `fact_sales` hapus-lalu-insert per `sk_date`.
- Hasil: **99.051** baris fact, 0 surrogate key NULL.

---
## 4. Skema Bintang (dwh)

![ERD Star Schema](docs/erd.png)

| Tabel | Peran | SCD | Baris |
|---|---|---|---:|
| `fact_sales` | Fact | – | **99.051** |
| `dim_time` | Waktu | Type 0 | 36.890 |
| `dim_product` | Produk | Type 1 | 452 |
| `dim_customer` | Pelanggan | Type 2 | 98.759 |
| `dim_employee` | Pegawai | Type 2 | 23 |

---
## 5. Pipeline Streaming — CDC MariaDB → Neon

Satu tabel `stg_categories` di MariaDB direplikasi **real-time** ke Postgres (Neon) memakai
**Debezium + Kafka Connect** (JDBC Sink), seperti tugas streaming.

```
MariaDB.stg_categories_manuel --[Debezium]--> Kafka --[JDBC Sink]--> Neon: stg.stg_categories_manuel
```

**Bukti replikasi berhasil (real-time):**

Sumber di MariaDB:

![Sumber MariaDB](docs/cdc1.jpeg)

Snapshot awal tersalin ke Neon:

![Snapshot Neon](docs/cdc2.jpeg)

INSERT & UPDATE di MariaDB:

![Perubahan MariaDB](docs/cdc3.jpeg)

Perubahan tercermin real-time di Neon (`77 TestCDC`, `99 Drone`):

![Hasil CDC Neon](docs/cdc4.jpeg)

Data lengkap di Neon:

![Data lengkap Neon](docs/cdc5.jpeg)

> Streaming ini **terpisah** dari batch (schema `stg`, tabel `stg_categories_manuel`) dan **tidak mengubah** `fact_sales` (tetap 99.051).

---
## 6. Visualisasi — Dashboard Metabase

![Dashboard Penjualan](docs/metabase.jpeg)

Semua metrik memakai **jumlah transaksi (count)** karena `total_price` di sumber = 0.

- **Total Transaksi: 99.051** — cocok dengan `fact_sales`.
- **Top 10 Produk** — terlaris berdekatan (~253–268), tidak ada yang mendominasi.
- **Top 10 Kota** — 1 negara (AS), 96 kota, sangat merata.
- **Tren per Bulan** — Jan 23.893 · Feb 21.420 · Mar 24.189 · Apr 22.635 · **Mei 6.914** (Mei belum penuh, data s/d 9 Mei).

---
## 7. Catatan Kualitas Data

1. `TotalPrice`=0 → dashboard memakai count, bukan revenue.
2. Rentang data Jan–9 Mei 2018 → Mei belum penuh; musiman belum valid.
3. Format tanggal tidak seragam → dinormalisasi sebelum SCD-2.
4. Idempotensi `fact_sales` terverifikasi (hapus-insert per `sk_date`).
5. Geografi homogen (1 negara/AS) → chart geografis per kota.

---
## 8. Kesimpulan

- **Dua pipeline berjalan**: batch (Airflow, 99.051 transaksi) + streaming (CDC real-time MariaDB→Neon).
- Skema bintang benar, SCD Type 0/1/2 sesuai mapping, relasi terbaca di Metabase.
- Penjualan merata antar produk & kota → portofolio sehat.